# C2. The symplectic transformation $g_\alpha$

Companion notebook to the bachelor's thesis *El formalismo ODM en sistemas híbridos clásico-cuánticos* (Santiago Puyol Miano, Universidad de Zaragoza, 2026).

The family of polarizations of the thesis comes from rotating the vertical polarization of $\Xi=T^*\mathbb R^2$ with a one-parameter subgroup $g_\alpha=e^{\alpha\xi}$ of $\mathrm{Sp}(4,\mathbb R)$. This notebook checks

1. that the generator $\xi$ lies in $\mathfrak{sp}(4,\mathbb R)$, that $g_\alpha=e^{\alpha\xi}$, and that $g_\alpha^\top Jg_\alpha=J$;
2. an identity for the pullback of $\Theta$ along $g_\alpha$;
3. why only the endpoints of the reparametrization $\alpha(\kappa)$ are fixed;
4. that $\hat x_q$ acts by multiplication at the von Neumann end;
5. what the operator algebra fixes along the family and what the polarization fixes.

$g_\alpha$ is a symplectic map. The κ-identification of C1 is a different map, and it is not symplectic.

**Kernel:** SageMath 10.8.

## Notation

- Coordinates $(x,p,\lambda_x,\lambda_p)$ on $\Xi$, in this order (indices 0 to 3), with $\Omega=dx\wedge d\lambda_x+dp\wedge d\lambda_p$ and $\Theta=\lambda_x\,dx+\lambda_p\,dp$.
- $\Omega(u,v)=u^\top Jv$ with

  ```
  J = [[ 0,  0,  1,  0],
       [ 0,  0,  0,  1],
       [-1,  0,  0,  0],
       [ 0, -1,  0,  0]]
  ```

  and $g\in\mathrm{Sp}(4,\mathbb R)$ if and only if $g^\top Jg=J$.
- $\alpha\in[0,\pi/2]$ is the rotation angle. The control parameter $\kappa\in[0,1]$ of ODM enters through a monotone reparametrization $\alpha=\alpha(\kappa)$ with $\alpha(0)=0$ and $\alpha(1)=\pi/2$ (section 3).
- A polarization is written as a $2\times4$ matrix whose rows span it. $P_0=\operatorname{span}\{\partial_{\lambda_x},\partial_{\lambda_p}\}$ is the vertical polarization (base $(x,p)$) and $P_{\pi/2}=\operatorname{span}\{\partial_{\lambda_x},\partial_p\}$ is the von Neumann polarization (base $(x,\lambda_p)$).

In [1]:
var('alpha')
assume(alpha >= 0, alpha <= pi/2)

# Omega(e_i, e_j) = J_ij in the basis (x, p, lambda_x, lambda_p)
J = matrix(SR, [
    [ 0,  0,  1,  0],
    [ 0,  0,  0,  1],
    [-1,  0,  0,  0],
    [ 0, -1,  0,  0]
])

assert (J + J.T).is_zero(),      "J must be skew-symmetric"
assert J.det() != 0,              "J must be non-degenerate"
print("OK: J is skew-symmetric, det J =", J.det())

OK: J is skew-symmetric, det J = 1


## 1. The generator $\xi$ and $g_\alpha=e^{\alpha\xi}$

The rotation that carries $P_0$ to $P_{\pi/2}$ fixes the pair $(x,\lambda_x)$ and rotates the conjugate pair $(p,\lambda_p)$:

```
ξ = [[ 0,  0, 0,  0],   x
     [ 0,  0, 0,  1],   p
     [ 0,  0, 0,  0],   λx
     [ 0, -1, 0,  0]]   λp
```

Since $\xi$ acts only on the $(p,\lambda_p)$ block and $\xi^2=-\Pi$, with $\Pi$ the projection onto that block, the exponential series sums to

```
g_α = (I − Π) + cos α · Π + sin α · ξ
    = [[1,      0, 0,     0],
       [0,  cos α, 0, sin α],
       [0,      0, 1,     0],
       [0, −sin α, 0, cos α]]
```

The cell checks $\xi^\top J+J\xi=0$, identifies $g_\alpha$ with $e^{\alpha\xi}$ through $g_0=I_4$ and $\tfrac{d}{d\alpha}g_\alpha=\xi g_\alpha$, and checks $g_\alpha^\top Jg_\alpha=J$.

In [2]:
# Generator: rotation in the (p, lambda_p) plane.
# Nonzero entries: xi[1,3] = 1 (row p, column lambda_p), xi[3,1] = -1 (row lambda_p, column p)
xi = matrix(SR, [
    [ 0,  0, 0,  0],
    [ 0,  0, 0,  1],
    [ 0,  0, 0,  0],
    [ 0, -1, 0,  0]
])

sp_cond = xi.T * J + J * xi
assert sp_cond.is_zero(), "xi not in sp(4): xi^T J + J xi != 0"
print("OK: ξᵀJ + Jξ = 0, so ξ ∈ sp(4,ℝ)")

# Closed form of exp(alpha*xi)
g = matrix(SR, [
    [1,           0, 0,          0],
    [0,  cos(alpha), 0, sin(alpha)],
    [0,           0, 1,          0],
    [0, -sin(alpha), 0, cos(alpha)]
])

# g = exp(alpha*xi) iff g(0) = I and dg/dalpha = xi*g
g0 = g.subs(alpha=0)
assert g0 == identity_matrix(SR, 4), f"g(0) != I: {g0}"
print("OK: g_0 = I₄")

dgda = matrix(SR, [[diff(g[i,j], alpha) for j in range(4)] for i in range(4)])
ode_res = dgda - xi * g
assert ode_res.is_zero(), f"ODE dg/dalpha = xi*g failed: {ode_res}"
print("OK: dg/dα = ξ·g, so g_α = exp(αξ)")

residual = g.T * J * g - J
residual_simp = residual.apply_map(lambda e: e.simplify_trig())
assert residual_simp.is_zero(), \
    f"g^T J g - J != 0; residual = {residual_simp}"
print("OK: g_αᵀ J g_α = J")

OK: ξᵀJ + Jξ = 0, so ξ ∈ sp(4,ℝ)
OK: g_0 = I₄
OK: dg/dα = ξ·g, so g_α = exp(αξ)
OK: g_αᵀ J g_α = J


## 2. The pullback of $\Theta$ along $g_\alpha$

The pullback of the tautological potential along $g_\alpha$ differs from $\Theta$ by an exact form:

$$g_\alpha^*\Theta-\Theta=d\tilde u_\alpha,\qquad \tilde u_\alpha=\tfrac12\sin\alpha\cos\alpha\,(\lambda_p^2-p^2)-\sin^2\alpha\;p\lambda_p,$$

with $\tilde u_0=0$ and $\tilde u_{\pi/2}=-p\lambda_p$.

This identity concerns $g_\alpha$ itself. The potential used in the thesis is $\theta_\alpha=(g_\alpha^{-1})^*\Theta=\Theta+du_\alpha$, the pullback along the inverse, which vanishes on the leaves of $P_\alpha$ for every $\alpha$ (checked in C6). The functions $\tilde u_\alpha$ and $u_\alpha$ agree at $\alpha=0$ and at $\alpha=\pi/2$, and differ in between.

In [3]:
var('x_v p_v lx_v lp_v', domain='real')   # coordinates of a point of Xi

# Theta = lambda_x dx + lambda_p dp evaluated at g_alpha(x, p, lambda_x, lambda_p):
# the lambda_x component stays lambda_x, the lambda_p component becomes -sin(alpha)*p + cos(alpha)*lambda_p
Theta_at_gq = vector(SR, [lx_v, -sin(alpha)*p_v + cos(alpha)*lp_v, 0, 0])

# Pullback along the linear map g: (g^*Theta)_j = sum_i Theta_i(g(pt)) * g_ij
gstar_Theta = Theta_at_gq * g   # coefficients on (dx, dp, dlambda_x, dlambda_p)

Theta_orig = vector(SR, [lx_v, lp_v, 0, 0])

difference = vector(SR, [(gstar_Theta[i] - Theta_orig[i]).simplify_trig() for i in range(4)])
print("g_α*Θ − Θ on (dx, dp, dλx, dλp):", list(difference))

u_tilde = (sin(alpha)*cos(alpha)/2)*(lp_v^2 - p_v^2) - sin(alpha)^2 * p_v * lp_v

du_tilde = vector(SR, [diff(u_tilde, v).simplify_trig()
                       for v in [x_v, p_v, lx_v, lp_v]])

residual = [(difference[i] - du_tilde[i]).simplify_trig() for i in range(4)]
assert all(r == 0 for r in residual), \
    f"g^*Theta - Theta != d(u_tilde); residual = {residual}"
print("OK: g_α*Θ − Θ = dũ_α")

u_at_0 = u_tilde.subs(alpha=0).simplify_full()
assert u_at_0.is_zero(), f"u_tilde at alpha=0 is {u_at_0}, expected 0"
print("OK: ũ_0 = 0")

u_at_pi2 = u_tilde.subs(alpha=pi/2).simplify_full()
assert (u_at_pi2 - (-p_v*lp_v)).simplify_full().is_zero(), \
    f"u_tilde at alpha=pi/2 is {u_at_pi2}, expected -p*lambda_p"
print("OK: ũ_{π/2} = −p λp, so g_{π/2}*Θ = λx dx − p dλp")

g_α*Θ − Θ on (dx, dp, dλx, dλp): [0, -p_v*cos(alpha)*sin(alpha) - lp_v*sin(alpha)^2, 0, lp_v*cos(alpha)*sin(alpha) - p_v*sin(alpha)^2]
OK: g_α*Θ − Θ = dũ_α
OK: ũ_0 = 0
OK: ũ_{π/2} = −p λp, so g_{π/2}*Θ = λx dx − p dλp


## 3. What fixes $\alpha(\kappa)$

The rotation angle $\alpha$ and the control parameter $\kappa$ are different parameters. Two facts decide how they are related.

1. The commutator $[\hat x_q,\hat p_q]=i\hbar\kappa$ follows, through the Kostant–Souriau prequantization, from the bracket $\{x_q,p_q\}_\Xi=\hbar\kappa$ of C1, and it holds for every $\alpha$. The operator algebra puts no condition on $\alpha(\kappa)$.
2. For $\kappa\neq0$, the Hamiltonian vector field of $x_q$ lies in $P_\alpha$ only when $\cos\alpha=0$, that is, at $\alpha=\pi/2$. So $\hat x_q$ acts by multiplication only at the von Neumann end.

Explicitly, $P_\alpha=\operatorname{span}\{\partial_{\lambda_x},\ \sin\alpha\,\partial_p+\cos\alpha\,\partial_{\lambda_p}\}$ and $X_{x_q}=-\tfrac{\hbar\kappa}{2}\partial_p-\partial_{\lambda_x}$. Writing $X_{x_q}=a\,\partial_{\lambda_x}+b\,(\sin\alpha\,\partial_p+\cos\alpha\,\partial_{\lambda_p})$ gives $a=-1$, $b=-\hbar\kappa/(2\sin\alpha)$ and $b\cos\alpha=0$, hence $\cos\alpha=0$.

The only conditions are $\alpha(0)=0$ and $\alpha(1)=\pi/2$. Any monotone reparametrization with these endpoints is admissible, and the thesis takes $\alpha(\kappa)=\pi\kappa/2$ as a convention. The cell checks the membership condition and the endpoint values of $g_\alpha$ and $P_\alpha$.

In [4]:
# P_alpha as a 2x4 matrix (defined here for the cells below)
P0_mat = matrix(SR, [[0, 0, 1, 0],
                      [0, 0, 0, 1]])
P_alpha = P0_mat * g.T    # P_alpha = P_0 * g_alpha^T

# X_{x_q} in P_alpha requires cos(alpha) = 0, whose only solution in [0, pi/2] is alpha = pi/2
assert SR(cos(pi/2)) == 0, "cos(pi/2) != 0"
print("OK: cos(π/2) = 0")

# At intermediate angles cos(alpha) > 0, so X_{x_q} is not in P_alpha
for test_alpha, expected in [(pi/6, sqrt(3)/2), (pi/4, sqrt(2)/2), (pi/3, SR(1)/2)]:
    val = SR(cos(test_alpha)).simplify_full()
    assert (val - expected).simplify_full() == 0, \
        f"cos({test_alpha}) mismatch: got {val}, expected {expected}"
    assert bool(val > 0), f"cos({test_alpha}) = {val} should be > 0"
    print(f"OK: cos({test_alpha}) = {val} > 0, so X_{{x_q}} is not in P_α there")

# Endpoint values
g_at_0 = g.subs(alpha=0)
assert g_at_0 == identity_matrix(SR, 4), f"g at alpha=0 is not I_4: {g_at_0}"
print("OK: g_0 = I₄ (κ = 0)")

g_end_expected = matrix(SR, [[1,0,0,0],[0,0,0,1],[0,0,1,0],[0,-1,0,0]])
g_at_pi2 = matrix(SR, [
    [g[i,j].subs(alpha=pi/2).simplify_trig() for j in range(4)]
    for i in range(4)
])
assert g_at_pi2 == g_end_expected, f"g at alpha=pi/2 mismatch: {g_at_pi2}"
print("OK: g_{π/2} sends p ↦ λp and λp ↦ −p (κ = 1)")

P_at_0 = matrix(SR, [
    [P_alpha[i,j].subs(alpha=0).simplify_trig() for j in range(4)]
    for i in range(2)
])
P0_ref = matrix(SR, [[0,0,1,0],[0,0,0,1]])
assert P_at_0 == P0_ref, f"P at alpha=0 is not P_0: {P_at_0}"
print("OK: P_0 = span{∂_λx, ∂_λp}, the vertical polarization")

P_vN_ref = matrix(SR, [[0,0,1,0],[0,1,0,0]])
P_at_pi2 = matrix(SR, [
    [P_alpha[i,j].subs(alpha=pi/2).simplify_trig() for j in range(4)]
    for i in range(2)
])
assert P_at_pi2 == P_vN_ref, f"P at alpha=pi/2 is not the von Neumann polarization: {P_at_pi2}"
print("OK: P_{π/2} = span{∂_λx, ∂_p}, the von Neumann polarization")

OK: cos(π/2) = 0
OK: cos(1/6*pi) = 1/2*sqrt(3) > 0, so X_{x_q} is not in P_α there
OK: cos(1/4*pi) = 1/2*sqrt(2) > 0, so X_{x_q} is not in P_α there
OK: cos(1/3*pi) = 1/2 > 0, so X_{x_q} is not in P_α there
OK: g_0 = I₄ (κ = 0)


OK: g_{π/2} sends p ↦ λp and λp ↦ −p (κ = 1)


OK: P_0 = span{∂_λx, ∂_λp}, the vertical polarization
OK: P_{π/2} = span{∂_λx, ∂_p}, the von Neumann polarization


## 4. $\hat x_q$ at the von Neumann end, and the midpoint $P_{\pi/4}$

The Hamiltonian vector field of $x_q=x-\tfrac{\hbar\kappa}{2}\lambda_p$ is $X_{x_q}=-\tfrac{\hbar\kappa}{2}\partial_p-\partial_{\lambda_x}$, which lies in $P_{\pi/2}=\operatorname{span}\{\partial_{\lambda_x},\partial_p\}$. The potential $\theta_{\pi/2}=\lambda_x\,dx-p\,d\lambda_p$ vanishes on it, because $dx$ and $d\lambda_p$ vanish on $\partial_p$ and $\partial_{\lambda_x}$. On sections $\psi(x,\lambda_p)$ the operator $\hat x_q$ is therefore multiplication by $x-\tfrac{\hbar\kappa}{2}\lambda_p$.

The cell also computes the tilted polarization at $\alpha=\pi/4$, which is $\kappa=1/2$ under $\alpha=\pi\kappa/2$, as an example of an intermediate point of the curve. This does not test the choice $\alpha(\kappa)=\pi\kappa/2$, which is a convention.

In [5]:
var('hbar kappa', domain='positive')  # physical hbar and control parameter kappa

# Hamiltonian vector field X_f = (df/dlambda_x) d_x + (df/dlambda_p) d_p - (df/dx) d_lambda_x - (df/dp) d_lambda_p
X_xq = vector(SR, [0, -hbar*kappa/2, -1, 0])   # components on (d_x, d_p, d_lambda_x, d_lambda_p)
print("X_{x_q} =", list(X_xq), " on (∂_x, ∂_p, ∂_λx, ∂_λp)")

# P_{pi/2} = span{d_lambda_x, d_p}: no d_x and no d_lambda_p component
assert X_xq[0] == 0, "X_{x_q} has a d_x component: not in P_{pi/2}"
assert X_xq[3] == 0, "X_{x_q} has a d_lambda_p component: not in P_{pi/2}"
print("OK: X_{x_q} ∈ P_{π/2} = span{∂_λx, ∂_p}")

# Midpoint alpha = pi/4
alpha_mid = pi/4
P_at_mid = matrix(SR, [
    [P_alpha[i,j].subs(alpha=alpha_mid).simplify_trig() for j in range(4)]
    for i in range(2)
])
expected_leaf2 = vector(SR, [0, 1/sqrt(2), 0, 1/sqrt(2)])
assert P_at_mid.row(0) == vector(SR, [0, 0, 1, 0]), \
    f"P_{{pi/4}} first leaf direction wrong: {P_at_mid.row(0)}"
diff_l2 = (P_at_mid.row(1) - expected_leaf2).apply_map(lambda e: e.simplify_full())
assert diff_l2.is_zero(), f"P_{{pi/4}} second leaf direction wrong: {P_at_mid.row(1)}"
print("OK: P_{π/4} = span{∂_λx, (∂_p + ∂_λp)/√2}")

X_{x_q} = [0, -1/2*hbar*kappa, -1, 0]  on (∂_x, ∂_p, ∂_λx, ∂_λp)
OK: X_{x_q} ∈ P_{π/2} = span{∂_λx, ∂_p}
OK: P_{π/4} = span{∂_λx, (∂_p + ∂_λp)/√2}


## 5. What the algebra fixes and what the polarization fixes

**The algebra.** Composing $x_q$ and $p_q$ with $g_\alpha^{-1}$ leaves their bracket equal to $\hbar\kappa$ for every $\alpha$, because $g_\alpha$ is symplectic. So $g_\alpha$, and the metaplectic operator $\mathcal U(g_\alpha)$ that implements it (C5), cannot produce the κ-dependence of the commutator. That dependence comes from the κ-identification.

**The polarization.** $P_\alpha$ decides which of $\hat x_q,\hat p_q$ act by multiplication. At $\alpha=0$ both $X_x$ and $X_p$ lie in $P_0=\operatorname{span}\{\partial_{\lambda_x},\partial_{\lambda_p}\}$, which is possible because $\{x,p\}_\Xi=0$. At $\alpha=\pi/2$, $X_{x_q}$ lies in $P_{\pi/2}$, and since $\{x_q,p_q\}_\Xi=\hbar\kappa\neq0$ the two operators cannot both act by multiplication. For $0<\alpha<\pi/2$, $\hat x_q$ is not a multiplication operator (section 3).

In [6]:
var('hbar kappa', domain='positive')
var('alpha_p', domain='positive')   # the angle as a positive symbol, separate from alpha above
var('xs ps lxs lps', domain='real')

def pb(f, g):
    # Poisson bracket {f,g}_Xi with Omega = dx^dlx + dp^dlp
    # {f,g} = df/dx*dg/dlx - df/dlx*dg/dx + df/dp*dg/dlp - df/dlp*dg/dp
    return (diff(f,xs)*diff(g,lxs) - diff(f,lxs)*diff(g,xs) +
            diff(f,ps)*diff(g,lps) - diff(f,lps)*diff(g,ps)).simplify_trig()

assert pb(xs, lxs) == 1, '{x,lx} != 1'
assert pb(ps, lps) == 1, '{p,lp} != 1'
assert pb(xs, ps)  == 0, '{x,p} != 0'
print('OK: {x, λx}_Ξ = {p, λp}_Ξ = 1 and {x, p}_Ξ = 0')

xq = xs - (hbar * kappa / 2) * lps
pq = ps + (hbar * kappa / 2) * lxs
brk_xqpq = pb(xq, pq)
assert brk_xqpq == hbar * kappa, 'bracket {x_q,p_q} wrong'
print('OK: {x_q, p_q}_Ξ = ħκ')

# The algebra: compose x_q, p_q with g_alpha^{-1}
# g_alpha^{-1}: (x,p,lx,lp) -> (x, p*cos(a)-lp*sin(a), lx, p*sin(a)+lp*cos(a))
p_pull  = ps * cos(alpha_p) - lps * sin(alpha_p)
lp_pull = ps * sin(alpha_p) + lps * cos(alpha_p)
xq_pull = xs  - (hbar * kappa / 2) * lp_pull
pq_pull = p_pull + (hbar * kappa / 2) * lxs

brk_pull = pb(xq_pull, pq_pull)
diff_brk = (brk_pull - hbar * kappa).simplify_trig()
assert diff_brk == 0, 'Bracket not preserved under g^{-1}'
print('OK: {x_q∘g_α⁻¹, p_q∘g_α⁻¹}_Ξ = ħκ for every α')

# The polarization.
# At alpha = 0, P_0 = span{d_lx, d_lp}: a vector is in P_0 iff its d_x (index 0) and d_p (index 1) components vanish.
X_x_comps = vector(SR, [0, 0, -1, 0])
X_p_comps = vector(SR, [0, 0, 0, -1])
assert X_x_comps[0] == 0 and X_x_comps[1] == 0, 'X_x has base components, not in P_0'
assert X_p_comps[0] == 0 and X_p_comps[1] == 0, 'X_p has base components, not in P_0'
print('OK: at α = 0, X_x = (0,0,−1,0) and X_p = (0,0,0,−1) lie in P_0, so x and p both act by multiplication')

# At alpha = pi/2, P_{pi/2} = span{d_lx, d_p}: a vector is in P_{pi/2} iff its d_x (index 0) and d_lp (index 3) components vanish.
# X_{x_q} = (dx_q/dlx, dx_q/dlp, -dx_q/dx, -dx_q/dp) = (0, -hbar*kappa/2, -1, 0)
X_xq_comps = vector(SR, [0, -hbar*kappa/2, -1, 0])
assert X_xq_comps[0] == 0, 'X_{x_q} has a d_x component: not in P_{pi/2}'
assert X_xq_comps[3] == 0, 'X_{x_q} has a d_lp component: not in P_{pi/2}'
print('OK: at α = π/2, X_{x_q} = (0,−ħκ/2,−1,0) lies in P_{π/2}, so x_q acts by multiplication')

OK: {x, λx}_Ξ = {p, λp}_Ξ = 1 and {x, p}_Ξ = 0
OK: {x_q, p_q}_Ξ = ħκ
OK: {x_q∘g_α⁻¹, p_q∘g_α⁻¹}_Ξ = ħκ for every α
OK: at α = 0, X_x = (0,0,−1,0) and X_p = (0,0,0,−1) lie in P_0, so x and p both act by multiplication
OK: at α = π/2, X_{x_q} = (0,−ħκ/2,−1,0) lies in P_{π/2}, so x_q acts by multiplication


In [7]:
# Every assert above has passed if this cell runs.
print('C2: all checks passed.')

C2: all checks passed.
